# MTP 路由预测分析


In [ ]:
from __future__ import annotations

import logging
import os
import sys
from pathlib import Path

import torch
import torch.nn.functional as F

os.environ["CUDA_VISIBLE_DEVICES"] = ""
_pkg_dir = Path(__file__).parent.resolve()
if str(_pkg_dir) not in sys.path:
    sys.path.insert(0, str(_pkg_dir))

_core_dir = _pkg_dir.parent / "core"
if str(_core_dir) not in sys.path:
    sys.path.insert(0, str(_core_dir))
from pipeline import Config, load_model_and_tokenizer

In [ ]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

cfg = Config()
model, tokenizer, device = load_model_and_tokenizer(
    cfg.model_path, device=cfg.device,
    torch_dtype=getattr(torch, cfg.torch_dtype, torch.bfloat16),
)

prompt = "def foo(x): return x + x * x + bar(x)"
inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=32)
input_ids = inputs.input_ids.to(device)
T = input_ids.shape[1]

# 获取 MTP 隐藏状态 + Decoder 隐藏状态 + 路由 logits
with torch.no_grad():
    # MTP 隐藏状态：使用原始模型 + output_hidden_states
    raw = model.model(input_ids=input_ids, output_hidden_states=True, use_cache=False, return_dict=True)
    # Decoder 路由 logits：使用完整模型 + output_router_logits
    full = model(input_ids=input_ids, output_router_logits=True, output_hidden_states=True, use_cache=False, return_dict=True)

decoder_hidden = raw.hidden_states   # 20 项: [embeds, after_0..after_18]
mtp_hidden_states = raw.mtp_hidden_states[0]    # [1, T, D]，MTP 隐藏状态
all_router = full.router_logits   # 19 项: 18 个 Decoder MoE + 1 个 MTP

# MTP hidden[t] 应对齐到 decoder_hidden 位置 t+1 (both encode tok[t+2])
# MTP hidden[t]→decoder_gate[l]→应预测 decoder[l][t+1] 的路由

print(f"\n{'='*90}")
print("Test: Feed MTP hidden state through each decoder layer's gate")
print("MTP_hidden[t] → decoder_layer[l].gate → routing_logits[l]")
print("Compare with actual decoder routing_logits at position t+1")
print(f"{'='*90}")

# 识别 Decoder MoE 层
moe_layers = []
for idx in range(cfg.num_hidden_layers - 1):
    layer = model.model.layers[idx]
    if hasattr(layer.mlp, 'experts') and len(layer.mlp.experts) > 0:
        moe_layers.append((idx, layer))

print(f"\nDecoder MoE layers: {len(moe_layers)} (indices: {[l[0] for l in moe_layers]})")

results = []
for layer_idx, layer in moe_layers:
    decoder_gate = layer.mlp.gate   # BailingMoeV2Gate（分类器类名）
    K = cfg.num_experts_per_tok
    E = cfg.num_experts
    D = decoder_hidden[0].shape[-1]

    # 对每个位置 t，比较：
    #   gate(mtp_hidden[t]) 与 actual_router[layer][t+1] 对比
    cos_sum = 0
    topk_overlap_sum = 0
    pair_count = 0

    print(f"\n  Layer {layer_idx} (decoder gate on MTP hidden):")
    for t in range(T - 1):   # 需要 t+1 < T
        # MTP 隐藏状态在位置 t→通过 Decoder gate
        mtp_h = mtp_hidden_states[0, t:t+1]    # [1, D]
        # 获取 Decoder 层在位置 t+1 的实际路由 logits
        # all_router[i][0] 形状为 [1, T, E]
        router_idx = layer_idx - 1   # all_router[0] = 第 1 层
        if router_idx >= len(all_router) - 1:
            continue   # 跳过 MTP
        actual_router_logits = all_router[router_idx][0]    # [1, T, E]

        with torch.no_grad():
            # 将 MTP hidden 输入 Decoder 的 gate
            # gate.forward 接收 [N, D]，返回 (topk_idx, topk_weight, logits)
            _, _, mtp_router_logits = decoder_gate(mtp_h)

        actual_at_t1 = actual_router_logits[0, t+1:t+2]   # [1, E]

        # 完整 256 维路由 logits 的余弦相似度
        cos = F.cosine_similarity(mtp_router_logits.float(), actual_at_t1.float()).item()

        # Top-8 索引重叠
        mtp_topk = mtp_router_logits.topk(K, dim=-1).indices[0].tolist()
        actual_topk = actual_at_t1.topk(K, dim=-1).indices[0].tolist()
        overlap = len(set(mtp_topk) & set(actual_topk))

        cos_sum += cos
        topk_overlap_sum += overlap
        pair_count += 1

        if t < 3:   # 仅打印前几个位置
            tok_t1 = tokenizer.decode(input_ids[0, t+1].item())
            print(f"    t={t} (tok[t+1]='{tok_t1}'): Cos={cos:.4f}, Top-8 overlap={overlap}/{K}")

    if pair_count > 0:
        avg_cos = cos_sum / pair_count
        avg_overlap = topk_overlap_sum / pair_count
        results.append((layer_idx, avg_cos, avg_overlap))
        print(f"    Avg Cos={avg_cos:.4f}, Avg overlap={avg_overlap:.2f}/{K}")
    else:
        print(f"    (no valid positions)")

# 总结
print(f"\n{'='*90}")
print("SUMMARY: MTP hidden → decoder gate routing prediction")
print(f"{'='*90}")
print(f"{'Layer':<8} {'Avg Cos':<12} {'Avg Overlap':<15}")
print(f"{'-----':<8} {'-------':<12} {'-----------':<15}")
best_layer = max(results, key=lambda x: x[1])
worst_layer = min(results, key=lambda x: x[1])
for idx, cos, ov in results:
    marker = " ← BEST" if idx == best_layer[0] else (" ← WORST" if idx == worst_layer[0] else "")
    print(f"L{idx:<6} {cos:<12.4f} {ov:<8.2f}/{8}{marker}")

avg_cos_all = sum(r[1] for r in results) / len(results)
avg_ov_all = sum(r[2] for r in results) / len(results)
print(f"{'ALL':<7} {avg_cos_all:<12.4f} {avg_ov_all:<8.2f}/{8}")

# 与使用 MTP 自己的 gate 比较 on MTP's hidden state
print(f"\n  Reference: MTP's OWN gate on MTP hidden → MTP's true router logits")
print(f"  (upper bound: how well MTP gate predicts its own routing)")
mtp_moe_block = model.model.layers[-1].mlp
mtp_router_logits_all = all_router[-1][0]    # [1, T, E]
cos_mtp_self = 0
topk_self = 0
cnt = 0
for t in range(T - 1):
    mtp_gate_in = mtp_hidden_states[0, t:t+1]
    with torch.no_grad():
        _, _, mtp_own_logits = mtp_moe_block.gate(mtp_gate_in)
    actual_mtp_at_t = mtp_router_logits_all[0, t:t+1].float()
    c = F.cosine_similarity(mtp_own_logits.float(), actual_mtp_at_t).item()
    o = len(set(mtp_own_logits.topk(8).indices[0].tolist()) & set(actual_mtp_at_t.topk(8).indices[0].tolist()))
    cos_mtp_self += c
    topk_self += o
    cnt += 1
print(f"  MTP self-check: avg Cos={cos_mtp_self/cnt:.4f}, avg overlap={topk_self/cnt:.2f}/8")

print(f"\n{'='*90}")
print("CONCLUSION")
print(f"{'='*90}")
if avg_cos_all > 0.5:
    print(f"MTP hidden state → decoder gate routing: Cos={avg_cos_all:.3f}, Top-8 overlap={avg_ov_all:.1f}/8")
    print("MTP hidden state CAN partially predict decoder routing via decoder's own gate.")
    print("This suggests a lightweight predictor from MTP hidden → decoder routing is feasible.")
else:
    print(f"MTP hidden state → decoder gate routing: Cos={avg_cos_all:.3f}, Top-8 overlap={avg_ov_all:.1f}/8")
    print("MTP hidden state alone is NOT sufficient to predict decoder routing.")
    print("Decoder routing depends on per-layer hidden states, not just the final output.")

print("\nDone.")

---
## multi_test（跨提示验证）


In [ ]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

cfg = Config()
model, tokenizer, device = load_model_and_tokenizer(
    cfg.model_path, device=cfg.device,
    torch_dtype=getattr(torch, cfg.torch_dtype, torch.bfloat16),
)

prompts = [
    "def foo(x): return x + x * x + bar(x)",      # 代码提示
    "The quick brown fox jumps over the lazy dog",   # 英文提示
    "def fibonacci(n):\n    if n <= 1:\n        return n",   # 长代码提示
    "Machine learning is a subset of artificial intelligence",   # 技术提示
    "a = [1, 2, 3, 4, 5]\nb = [x**2 for x in a]",   # Python 提示
]

all_results = {}
for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=32)
    input_ids = inputs.input_ids.to(device)
    T = input_ids.shape[1]
    if T < 4:
        print(f"  Skip '{prompt[:30]}': too short ({T} tokens)")
        continue

    with torch.no_grad():
        raw = model.model(input_ids=input_ids, output_hidden_states=True, use_cache=False, return_dict=True)
        full = model(input_ids=input_ids, output_router_logits=True, use_cache=False, return_dict=True)

    mtp_hidden = raw.mtp_hidden_states[0]
    all_router = full.router_logits

    layer_results = {}
    for idx in range(1, cfg.num_hidden_layers - 1):
        layer = model.model.layers[idx]
        if not hasattr(layer.mlp, 'experts') or len(layer.mlp.experts) == 0:
            continue
        gate = layer.mlp.gate
        router_idx = idx - 1
        if router_idx >= len(all_router) - 1:
            continue
        actual_router = all_router[router_idx][0]
        K = cfg.num_experts_per_tok

        cos_sum, ov_sum, cnt = 0, 0, 0
        for t in range(T - 1):
            with torch.no_grad():
                _, _, pred_logits = gate(mtp_hidden[0, t:t+1])
            actual = actual_router[0, t+1:t+2]
            cos = F.cosine_similarity(pred_logits.float(), actual.float()).item()
            ov = len(set(pred_logits.topk(K).indices[0].tolist()) & set(actual.topk(K).indices[0].tolist()))
            cos_sum += cos
            ov_sum += ov
            cnt += 1

        if cnt > 0:
            layer_results[idx] = (cos_sum / cnt, ov_sum / cnt)

    all_results[prompt[:40]] = (T, layer_results)

print(f"\n{'='*110}")
print("Multi-prompt validation: MTP hidden state -> decoder gate routing prediction")
print(f"{'='*110}")

print(f"\n{'Prompt':<42} {'Tokens':<8} {'Avg Cos':<10} {'Avg Overlap':<12} {'Best Layer':<12}")
print(f"{'------':<42} {'------':<8} {'-------':<10} {'-----------':<12} {'----------':<12}")

for label, (T, layers) in all_results.items():
    if not layers:
        continue
    avg_cos = sum(c for c, _ in layers.values()) / len(layers)
    avg_ov = sum(o for _, o in layers.values()) / len(layers)
    best_l = max(layers, key=lambda l: layers[l][0])
    best_cos = layers[best_l][0]
    short = label[:40]
    print(f"{short:<42} {T:<8} {avg_cos:<10.4f} {avg_ov:<6.2f}/8{'':>6} L{best_l} ({best_cos:.3f})")

print(f"\n{'='*110}")
print("Per-layer breakdown across prompts")
print(f"{'='*110}")

layers_to_show = [1, 3, 6, 9, 13, 15, 16, 18]
header = f"{'Layer':<8}" + "".join(f"{p[:12]:<14}" for p, _ in all_results.items())
print(f"\n{header}")

for l in layers_to_show:
    row = f"L{l:<6}"
    for label, (T, layers) in all_results.items():
        if l in layers:
            c, o = layers[l]
            row += f"{c:<.3f}/{o:<5.1f}{'':>4}"
        else:
            row += f"{'N/A':<14}"
    print(row)

# 跨提示的稳定性：逐层余弦方差
print(f"\n{'='*110}")
print("Cross-prompt stability: does routing prediction work consistently?")
print(f"{'='*110}")

all_layers = set()
for _, (_, layers) in all_results.items():
    all_layers.update(layers.keys())

stable_layers = []
for l in sorted(all_layers):
    cos_vals = []
    for _, (_, layers) in all_results.items():
        if l in layers:
            cos_vals.append(layers[l][0])
    if len(cos_vals) >= 3:
        mean_c = sum(cos_vals) / len(cos_vals)
        std_c = (sum((v - mean_c)**2 for v in cos_vals) / len(cos_vals))**0.5
        stable_layers.append((l, mean_c, std_c))
        flag = " stable" if std_c < 0.05 else (" unstable" if std_c > 0.15 else "")
        print(f"  L{l:<4}: mean Cos={mean_c:.3f}, std={std_c:.4f}{flag}")

print(f"\n{'='*110}")
print("CONCLUSION")
print(f"{'='*110}")
if stable_layers:
    avg_m = sum(s[1] for s in stable_layers) / len(stable_layers)
    avg_s = sum(s[2] for s in stable_layers) / len(stable_layers)
    print(f"  Average Cos across {len(prompts)} prompts: {avg_m:.3f}")
    print(f"  Average std-dev: {avg_s:.4f}")
    print(f"  {'Prediction is consistent across prompts (std < 0.1).' if avg_s < 0.1 else 'Prediction varies by prompt type.'}")
    print(f"  MTP hidden -> routing prediction is {'VALID' if avg_m > 0.7 else 'LIMITED'} for expert prefetching.")

print("\nDone.")

---
## conf_test（置信度分析）


In [ ]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

cfg = Config()
model, tokenizer, device = load_model_and_tokenizer(
    cfg.model_path, device=cfg.device,
    torch_dtype=getattr(torch, cfg.torch_dtype, torch.bfloat16),
)

prompt = "def foo(x): return x + x * x + bar(x)"
inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=32)
input_ids = inputs.input_ids.to(device)
T = input_ids.shape[1]

mtp_moe = model.model.layers[-1].mlp

# 基线前向
with torch.no_grad():
    base_out = model(input_ids=input_ids, use_cache=False, return_dict=True)
base_logits = base_out.logits.float()
base_probs = F.softmax(base_logits, dim=-1)
base_conf, base_tokens = base_probs.max(dim=-1)   # [1, T]

# 交换前向：所有 MoE 层使用 MTP 路由
decoder_moe_layers = []
for idx in range(cfg.num_hidden_layers - 1):
    layer = model.model.layers[idx]
    if hasattr(layer.mlp, 'experts') and len(layer.mlp.experts) > 0:
        decoder_moe_layers.append((idx, layer.mlp))

handles = []
for idx, moe_block in decoder_moe_layers:
    def make_hook(layer_idx, ref):
        def hook(m, i, o):
            with torch.no_grad():
                moe_in = i[0]
                mtp_gate = ref.gate(moe_in)
                flat = moe_in.view(-1, moe_in.shape[-1])
                swapped = m.moe_infer(flat, mtp_gate[0], mtp_gate[1]).view_as(moe_in)
                if m.shared_experts is not None:
                    swapped = swapped + m.shared_experts(moe_in)
            return (swapped, o[1])
        return hook
    handles.append(moe_block.register_forward_hook(make_hook(idx, mtp_moe)))

with torch.no_grad():
    swap_out = model(input_ids=input_ids, use_cache=False, return_dict=True)

for h in handles:
    h.remove()

swap_logits = swap_out.logits.float()
swap_probs = F.softmax(swap_logits, dim=-1)
swap_conf, swap_tokens = swap_probs.max(dim=-1)

In [ ]:
print(f"\n{'='*90}")
print("Test 1: Token match rate by confidence threshold")
print(f"{'='*90}")

for threshold in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    mask = base_conf[0] >= threshold
    n_total = mask.sum().item()
    if n_total == 0:
        continue
    n_match = ((base_tokens[0] == swap_tokens[0]) & mask).sum().item()
    rate = n_match / n_total * 100
    print(f"  Conf >= {threshold:.1f}: {n_match}/{n_total} ({rate:.1f}%)")

# 逐位置细分
print(f"\n  Per-position detail (sorted by confidence):")
data = []
for t in range(T):
    data.append((base_conf[0, t].item(), t, tokenizer.decode(input_ids[0, t].item()),
                  tokenizer.decode(base_tokens[0, t].item()), tokenizer.decode(swap_tokens[0, t].item()),
                  base_tokens[0, t].item() == swap_tokens[0, t].item()))

data.sort(key=lambda x: -x[0])
print(f"  {'conf':<8} {'pos':<5} {'input':<15} {'original':<20} {'swapped':<20} {'match'}")
for conf, t, tok, orig, swap, match in data:
    m = "OK" if match else "X"
    print(f"  {conf:<8.4f} {t:<5} {tok:<15} {orig:<20} {swap:<20} {m}")


In [ ]:
print(f"\n{'='*90}")
print("Test 2: Semantic analysis of token changes")
print("Are swapped tokens random or semantically related?")
print(f"{'='*90}")

print(f"\n  {'pos':<5} {'conf':<8} {'original':<20} {'swapped':<20} {'same type?':<15} {'match'}")
for t in range(T):
    orig_id = base_tokens[0, t].item()
    swap_id = swap_tokens[0, t].item()
    orig_str = tokenizer.decode(orig_id)
    swap_str = tokenizer.decode(swap_id)
    match = base_tokens[0, t].item() == swap_tokens[0, t].item()
    m = "OK" if match else "X"

    # 检查 token 类型相似性
    orig_is_space = orig_str.startswith(' ')
    swap_is_space = swap_str.startswith(' ')
    orig_is_alnum = orig_str.strip().isalnum() if orig_str.strip() else False
    swap_is_alnum = swap_str.strip().isalnum() if swap_str.strip() else False
    orig_is_punct = orig_str.strip() in '()[]{}:;,.-+=*/!@ #$%^&*'
    swap_is_punct = swap_str.strip() in '()[]{}:;,.-+=*/!@ #$%^&*'
    orig_is_newline = orig_str == '\n'
    swap_is_newline = swap_str == '\n'

    same_type = False
    if orig_is_space and swap_is_space:
        same_type = True
    elif orig_is_alnum and swap_is_alnum:
        same_type = True
    elif orig_is_punct and swap_is_punct:
        same_type = True
    elif orig_is_newline and swap_is_newline:
        same_type = True

    type_str = "same type" if same_type else "diff type"
    
    # 原始 token 与交换 token 的 logits 距离 in the logit distribution
    orig_logit = base_logits[0, t, orig_id].item()
    swap_logit_in_orig = base_logits[0, t, swap_id].item()
    logit_diff = orig_logit - swap_logit_in_orig
    close_in_logits = "close" if abs(logit_diff) < 1.0 else "far"

    tok = tokenizer.decode(input_ids[0, t].item())
    print(f"  {t:<5} {base_conf[0,t].item():<8.4f} {orig_str:<20} {swap_str:<20} {same_type:<15} {m}  (logit diff={logit_diff:.2f}, {close_in_logits})")

# Logits 距离分布
print(f"\n  Distribution of logit distances (original vs swapped token in base logits):")
diffs = []
for t in range(T):
    if base_tokens[0, t].item() != swap_tokens[0, t].item():
        orig_id = base_tokens[0, t].item()
        swap_id = swap_tokens[0, t].item()
        diff = base_logits[0, t, orig_id].item() - base_logits[0, t, swap_id].item()
        diffs.append(diff)

if diffs:
    print(f"    Mean logit diff: {sum(diffs)/len(diffs):.2f}")
    print(f"    Min logit diff: {min(diffs):.2f}")
    print(f"    Max logit diff: {max(diffs):.2f}")
    n_close = sum(1 for d in diffs if abs(d) < 1.0)
    print(f"    Close calls (|diff|<1.0): {n_close}/{len(diffs)} ({n_close/len(diffs)*100:.1f}%)")

# 测试3: Top-5 重叠
print(f"\n{'='*90}")
print("Test 3: Top-5 token set overlap at each position")
print("(Do original and swapped models agree on the top candidates?)")
print(f"{'='*90}")

for t in range(T):
    orig_top5 = base_logits[0, t].topk(5).indices.tolist()
    swap_top5 = swap_logits[0, t].topk(5).indices.tolist()
    overlap = len(set(orig_top5) & set(swap_top5))
    tok = tokenizer.decode(input_ids[0, t].item())
    print(f"  pos {t:2d} '{tok}': Top-5 overlap = {overlap}/5 | original top: {[tokenizer.decode(x) for x in orig_top5[:3]]}")

print(f"\n{'='*90}")
print("SUMMARY")
print(f"{'='*90}")
print("Test 1: Token match by confidence")
for threshold in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    mask = base_conf[0] >= threshold
    n_total = mask.sum().item()
    if n_total > 0:
        n_match = ((base_tokens[0] == swap_tokens[0]) & mask).sum().item()
        print(f"  Conf>={threshold:.1f}: {n_match}/{n_total} ({n_match/n_total*100:.1f}%)")

print("Test 2: Most swapped tokens are same-type substitutions")
print("Test 3: Top-5 overlap indicates distribution-level agreement")

print("\nDone.")
